# Agente Car con ambiente Space

Código con ejemplo básico de agente para simular un automóvil. Se deben implementar funciones para interacción, evitar colisiones y permitir vueltas y reglas requeridas por el reto.

In [1]:
!pip install agentpy
# Model design
import agentpy as ap
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import IPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.9/778.9 kB 17.4 MB/s eta 0:00:00


## Definición del modelo

In [2]:
class Car(ap.Agent):
    """ An agent with a position and velocity in a continuous space. """
    def setup(self):
        self.space = self.model.space
        self.velocity = [0, 0]

    def update_position(self):
        # Aleatoriamente acelera, frena, o nada
        if np.random.rand() < .5:
            self.acelera() if np.random.rand() < .5 else self.frena()
        self.space.move_by(self, self.velocity)

    def get_position(self):
        return self.space.positions[self]

    # Acelera un 5% de la velocidad actual
    def acelera(self):
        if np.sum(self.velocity) == 0:
            self.velocity = [.2,0] if self.get_position()[0] == 0 else [0,.2]
        else:
            self.velocity[0] *= 1.05
            self.velocity[1] *= 1.05

    # Frena un 5% de la velocidad actual
    def frena(self):
        self.velocity[0] *= .95
        self.velocity[1] *= .95

In [3]:
class CarModel(ap.Model):

    def setup(self):
        # Inicializamos parametros del modelo
        self.space = ap.Space(self, shape=(self.p.size, self.p.size))

    def step(self):
        # De acuerdo a la probabilidad de aparición, se genera un carro.
        if np.random.rand() < self.p.prob:
            # Aleatoriamente se agrega en vertical u horizontal
            r = np.random.randint((self.p.size - self.p.street_width) // 2, (self.p.size + self.p.street_width) // 2)
            pos = [np.array([0., r])] if np.random.rand() < .5 else [np.array([r, 0.])]
            self.space.add_agents(ap.AgentList(self, 1, Car), positions=pos)

        # Actualizar posición
        self.space.agents.update_position()

        # Eliminar agentes que salen
        remove = []
        for car in self.space.agents:
            pos = car.get_position()
            if pos[0] < 0 or pos[0] >= self.p.size or pos[1] < 0 or pos[1] >= self.p.size:
                remove.append(car)

        self.space.remove_agents(remove)

## Visualización

In [5]:
def animation_plot_single(m, ax):
    ax.set_title(f"Cars Model t={m.t}")
    #pos = m.space.positions.values()
    #pos = np.array(list(pos)).T  # Transform
    arriba = np.array(list(map(lambda a: a.get_position(), filter(lambda x: x.velocity[0] == 0, m.space.agents)))).T
    if len(arriba) > 0:
        ax.scatter(*arriba, s=30, marker='^', c='black')
    derecha = np.array(list(map(lambda a: a.get_position(), filter(lambda x: x.velocity[1] == 0, m.space.agents)))).T
    if len(derecha) > 0:
        ax.scatter(*derecha, s=30, marker='<', c='black')
    ax.set_xlim(0, m.p.size-1)
    ax.set_ylim(0, m.p.size-1)
    #ax.set_axis_off()

def animation_plot(m, p):
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(111)
    animation = ap.animate(m(p), fig, ax, animation_plot_single)
    return IPython.display.HTML(animation.to_jshtml(fps=20))

## Simulación

To run a simulation, we define a dictionary with our parameters:

In [6]:
parameters = {
    'size': 20,
    'street_width': 2,
    'steps': 200,
    'seed': 123,
    'prob': .1
}

animation_plot(CarModel, parameters)